# Score Margin Filter: Robustness Analysis

The single-path bankroll simulation showed [-1,+1] at 672x and [-3,+3] at 249x. But these are
single sequences through movie-by-movie compounding — one lucky/unlucky ordering changes everything.

**Method:** Bootstrap resampling. Shuffle movie order (sample with replacement) 10,000 times per
configuration, simulate each, and look at the *distribution* of outcomes.

**What we want to know:**
- Median multiplier (robust center, not skewed by lucky runs)
- 5th/25th percentile (downside when unlucky)
- Std dev (how variable is the outcome)
- P(ruin) (how often does bankroll hit zero)
- Which config has the best risk-adjusted outcome we'd actually be comfortable betting on

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

trades = pd.read_csv("/tmp/claude/trades_cache.csv")
trades["snapshot_time"] = pd.to_datetime(trades["snapshot_time"], utc=True)
print(f"Loaded {len(trades):,} evaluations, {trades['slug'].nunique()} movies")

In [ ]:
def get_movie_results(trades_df, min_edge, margin_floor=None, margin_ceil=None):
    """Extract per-movie P&L results for a given strategy configuration.
    Returns a DataFrame with one row per movie: total_pnl, total_cost, n_positions.
    """
    ACTION_WINDOW = (24, 120)
    mask = (
        (trades_df["direction"] == "No") &
        (trades_df["abs_edge"] >= min_edge) &
        (trades_df["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades_df["hours_to_close"] <= ACTION_WINDOW[1])
    )
    if margin_floor is not None:
        mask &= (trades_df["score_margin"] >= margin_floor)
    if margin_ceil is not None:
        mask &= (trades_df["score_margin"] <= margin_ceil)
    
    no_trades = trades_df[mask].sort_values("snapshot_time")
    positions = no_trades.groupby(["slug", "threshold"]).first().reset_index()
    positions["entry_cost"] = 100 - positions["market_price"]
    positions["pos_pnl"] = np.where(
        ~positions["resolved_yes"], positions["market_price"], -positions["entry_cost"])
    
    movie_results = positions.groupby("slug").agg(
        total_pnl=("pos_pnl", "sum"),
        total_cost=("entry_cost", "sum"),
        n_positions=("pos_pnl", "count"),
    ).reset_index()
    
    # ROI per movie (pnl / cost)
    movie_results["roi"] = movie_results["total_pnl"] / movie_results["total_cost"]
    movie_results["won"] = movie_results["total_pnl"] > 0
    return movie_results


def simulate_bankroll_from_movies(movie_results, bankroll_frac, start_bankroll=100000.0):
    """Run bankroll simulation given a (possibly resampled) sequence of movie results."""
    bankroll = start_bankroll
    for _, movie in movie_results.iterrows():
        risk_budget = bankroll * bankroll_frac
        contracts_scale = risk_budget / movie["total_cost"] if movie["total_cost"] > 0 else 0
        bankroll += movie["total_pnl"] * contracts_scale
        if bankroll <= 0:
            return 0.0
    return bankroll


def bootstrap_bankroll(movie_results, bankroll_frac, n_sims=10000,
                       start_bankroll=100000.0, rng_seed=42):
    """Bootstrap: resample movies with replacement, simulate each, return distribution."""
    rng = np.random.default_rng(rng_seed)
    n_movies = len(movie_results)
    finals = np.zeros(n_sims)
    
    for i in range(n_sims):
        # Resample movies with replacement, same count
        idx = rng.choice(n_movies, size=n_movies, replace=True)
        resampled = movie_results.iloc[idx].reset_index(drop=True)
        finals[i] = simulate_bankroll_from_movies(resampled, bankroll_frac, start_bankroll)
    
    return finals

print("Functions defined.")

## Per-movie P&L distributions

Before bootstrapping, look at the actual per-movie wins and losses for each config.

In [ ]:
configs = [
    (10, None, None, "10c, no filter"),
    (15, None, None, "15c, no filter"),
    (20, None, None, "20c, no filter"),
    (10, -3, 3, "10c, [-3,+3]"),
    (15, -3, 3, "15c, [-3,+3]"),
    (20, -3, 3, "20c, [-3,+3]"),
    (15, -1, 1, "15c, [-1,+1]"),
    (20, -1, 1, "20c, [-1,+1]"),
]

all_movie_results = {}
for me, floor, ceil, label in configs:
    mr = get_movie_results(trades, me, margin_floor=floor, margin_ceil=ceil)
    all_movie_results[label] = mr
    
    wins = mr["won"].sum()
    losses = len(mr) - wins
    avg_win = mr.loc[mr["won"], "roi"].mean() if wins > 0 else 0
    avg_loss = mr.loc[~mr["won"], "roi"].mean() if losses > 0 else 0
    print(f"{label:<20s}  movies={len(mr):>3d}  W/L={wins}/{losses}  "
          f"avg_win_roi={avg_win:+.1%}  avg_loss_roi={avg_loss:+.1%}  "
          f"median_roi={mr['roi'].median():+.1%}")

In [ ]:
# Per-movie ROI distributions: histograms
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharey=True)
axes = axes.flatten()

for i, (label, mr) in enumerate(all_movie_results.items()):
    ax = axes[i]
    roi_pct = mr["roi"] * 100
    bins = np.arange(-150, 200, 10)
    ax.hist(roi_pct[mr["won"]], bins=bins, color="green", alpha=0.7, label="Winners")
    ax.hist(roi_pct[~mr["won"]], bins=bins, color="red", alpha=0.7, label="Losers")
    ax.axvline(0, color="k", linewidth=0.5)
    ax.axvline(roi_pct.median(), color="blue", linewidth=1.5, linestyle="--",
              label=f"median={roi_pct.median():.0f}%")
    ax.set_title(f"{label}\n({len(mr)} movies, {mr['won'].mean():.0%} WR)", fontsize=10)
    ax.set_xlabel("ROI (%)")
    ax.legend(fontsize=7)

fig.suptitle("Per-Movie ROI Distribution by Strategy", fontsize=13)
plt.tight_layout()
plt.show()

## Bootstrap simulation (10,000 resamples per config)

In [ ]:
FRAC = 0.10
START = 100000
N_SIMS = 10000

bootstrap_results = {}
summary_rows = []

for label, mr in all_movie_results.items():
    print(f"Bootstrapping {label}...", flush=True)
    finals = bootstrap_bankroll(mr, FRAC, n_sims=N_SIMS, start_bankroll=START)
    bootstrap_results[label] = finals
    
    mults = finals / START
    summary_rows.append({
        "config": label,
        "movies": len(mr),
        "p5": np.percentile(mults, 5),
        "p25": np.percentile(mults, 25),
        "median": np.median(mults),
        "p75": np.percentile(mults, 75),
        "p95": np.percentile(mults, 95),
        "mean": np.mean(mults),
        "std": np.std(mults),
        "p_ruin": (finals <= 0).mean(),
        "p_loss": (mults < 1).mean(),
    })

summary = pd.DataFrame(summary_rows)
print("\n=== Bootstrap Summary (10K resamples, 10% risk/movie) ===")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

In [ ]:
# Distribution plots: log-scale bankroll multiplier
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, (label, finals) in enumerate(bootstrap_results.items()):
    ax = axes[i]
    mults = finals / START
    log_mults = np.log10(np.maximum(mults, 0.01))  # floor at 0.01x for log
    
    ax.hist(log_mults, bins=50, color="steelblue", alpha=0.7, edgecolor="white")
    ax.axvline(np.log10(np.median(mults)), color="red", linewidth=2,
              label=f"median={np.median(mults):.0f}x")
    ax.axvline(np.log10(max(np.percentile(mults, 5), 0.01)), color="orange",
              linewidth=1.5, linestyle="--", label=f"p5={np.percentile(mults, 5):.0f}x")
    ax.axvline(0, color="k", linewidth=0.5, linestyle=":")
    ax.set_title(f"{label}\n({all_movie_results[label].shape[0]} movies)", fontsize=10)
    ax.set_xlabel("log10(multiplier)")
    ax.legend(fontsize=7)

fig.suptitle("Bootstrap Bankroll Multiplier Distributions (10K resamples)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side box plots (log scale)
fig, ax = plt.subplots(figsize=(14, 6))

labels = list(bootstrap_results.keys())
data = [np.log10(np.maximum(bootstrap_results[l] / START, 0.01)) for l in labels]

bp = ax.boxplot(data, labels=labels, vert=True, patch_artist=True,
                showfliers=False, whis=[5, 95])

colors = ["#4c72b0"] * 3 + ["#55a868"] * 3 + ["#c44e52"] * 2
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.axhline(0, color="k", linewidth=0.5, linestyle=":", label="Break even (1x)")
ax.set_ylabel("log10(multiplier)")
ax.set_title("Bankroll Multiplier Distribution by Strategy (whiskers=5th/95th pct)")
ax.tick_params(axis="x", rotation=30)

# Add median annotation
for i, label in enumerate(labels):
    med = np.median(bootstrap_results[label] / START)
    ax.text(i + 1, np.log10(max(med, 0.01)) + 0.15, f"{med:.0f}x",
            ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

## Downside focus: worst-case analysis

For each config, what does the bottom 10% of outcomes look like?

In [ ]:
print("=== Downside Analysis ===")
print(f"{'config':<20s} {'p1':>6s} {'p5':>6s} {'p10':>6s} {'p25':>6s} {'p_loss':>7s} {'p_ruin':>7s}")
print("-" * 60)

for label, finals in bootstrap_results.items():
    mults = finals / START
    print(f"{label:<20s} {np.percentile(mults, 1):>5.1f}x {np.percentile(mults, 5):>5.1f}x "
          f"{np.percentile(mults, 10):>5.1f}x {np.percentile(mults, 25):>5.1f}x "
          f"{(mults < 1).mean():>6.1%} {(finals <= 0).mean():>6.1%}")

In [ ]:
# Sharpe-like ratio: median / std of log-multiplier
# Higher = more reward per unit of outcome variance
print("=== Risk-adjusted metrics ===")
print(f"{'config':<20s} {'median':>8s} {'mean':>8s} {'std':>8s} {'med/std':>8s} {'mean/std':>8s}")
print("-" * 58)

for label, finals in bootstrap_results.items():
    mults = finals / START
    log_m = np.log10(np.maximum(mults, 0.01))
    print(f"{label:<20s} {np.median(mults):>7.0f}x {np.mean(mults):>7.0f}x "
          f"{np.std(mults):>7.0f}x {np.median(mults)/np.std(mults):>7.2f} "
          f"{np.mean(mults)/np.std(mults):>7.2f}")